# Shallow Diffuse 主表 baseline 单独真实 GPU 链路入口

该 Notebook 只运行 `shallow_diffuse` 的 SD3.5 common-backbone method-faithful 主表轨道。三个 baseline 可在独立 Colab GPU 会话中并行执行, 结果通过各自 transfer manifest 进入 CPU 论文闭合。

Notebook 仅以 `python -I` 调用统一宿主 launcher 并读取受治理结果, 不承载 baseline 算法、阈值、攻击或统计实现。

## 运行前准备

1. 在 Colab 中选择 GPU runtime。
2. 配置可访问 `stabilityai/stable-diffusion-3.5-medium` 的 `HF_TOKEN`。
3. 选择 `probe_paper`、`pilot_paper` 或 `full_paper` 运行层级。
4. 本入口生成 `split_observations/shallow_diffuse_baseline_observations.json` 与对应 transfer manifest。
5. 结果包只有在真实执行、fixed-FPR 阈值和完整攻击集合全部通过时才会生成。
6. Notebook 不在宿主解释器准备 profile; 精确 `workflow_orchestrator` 子解释器会创建并核验 `sd35_method_runtime_gpu` 隔离科学子解释器.

运行期间, repository 共享持久化会话会把稳定文件周期写入当前 workflow 的 Drive 目录, 并在恢复前复验 formal execution commit、科学 profile、配置身份与逐文件 SHA-256. checkpoint 仅用于续跑, 不属于正式论文证据。
7. 每次会话必须显式选择一个登记的 `SLM_WM_RANDOMIZATION_REPEAT_ID`; 权威9个 seed-key repeat 分别运行并写入独立持久化目录。


In [ ]:
SLM_WM_PAPER_RUN_NAME = "probe_paper"
SLM_WM_RANDOMIZATION_REPEAT_ID = "seed_00_key_00"

import os

from google.colab import drive

drive.mount('/content/drive')
os.environ["SLM_WM_PAPER_RUN_NAME"] = SLM_WM_PAPER_RUN_NAME
os.environ["SLM_WM_RANDOMIZATION_REPEAT_ID"] = SLM_WM_RANDOMIZATION_REPEAT_ID
SLM_WM_PRIMARY_BASELINE_ID = "shallow_diffuse"
os.environ["SLM_WM_PRIMARY_BASELINE_ID"] = SLM_WM_PRIMARY_BASELINE_ID


In [ ]:
import os
import re
import subprocess
import sys
from pathlib import Path

repository_url = os.environ.get("SLM_WM_REPOSITORY_URL", "https://github.com/RICHAAARC/SLM-WM.git")
repository_commit = os.environ.get("SLM_WM_REPOSITORY_COMMIT", "")
if re.fullmatch(r"[0-9a-f]{40}", repository_commit) is None:
    raise ValueError("正式运行前必须通过 SLM_WM_REPOSITORY_COMMIT 提供精确40位小写 Git SHA")
workspace_dir = Path(os.environ.get("SLM_WM_WORKSPACE_DIR", "/content/slm_wm_repository"))
workspace_dir.parent.mkdir(parents=True, exist_ok=True)
if workspace_dir.exists() and not (workspace_dir / ".git").is_dir():
    raise RuntimeError("SLM_WM_WORKSPACE_DIR 已存在但不是 Git 仓库")
if not (workspace_dir / ".git").is_dir():
    subprocess.run(["git", "clone", repository_url, str(workspace_dir)], check=True)

status_before_checkout = subprocess.run(
    ["git", "status", "--porcelain=v1", "--untracked-files=all"],
    cwd=workspace_dir,
    check=True,
    capture_output=True,
    text=True,
).stdout
if status_before_checkout:
    raise RuntimeError("复用 Colab 工作区前必须先清理 Git 工作树")
subprocess.run(["git", "fetch", "origin", "--tags", "--prune"], cwd=workspace_dir, check=True)
subprocess.run(["git", "rev-parse", "--verify", repository_commit + "^{commit}"], cwd=workspace_dir, check=True)
subprocess.run(["git", "checkout", "--detach", repository_commit], cwd=workspace_dir, check=True)

os.chdir(workspace_dir)
print({"workspace_dir": str(workspace_dir), "repository_commit": repository_commit})


In [ ]:
FORMAL_HOST_LAUNCHER = "scripts/run_formal_workflow_host.py"
print({"formal_host_launcher": FORMAL_HOST_LAUNCHER})


In [ ]:
# 论文运行配置由精确 workflow_orchestrator 子解释器统一加载.


In [ ]:
import os

try:
    from google.colab import userdata
    token_from_secret = userdata.get('HF_TOKEN')
except Exception:
    token_from_secret = None

if token_from_secret and not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = token_from_secret

print({'hf_token_available_for_scientific_child': bool(os.environ.get('HF_TOKEN'))})


In [ ]:
gpu_probe = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    check=False,
    capture_output=True,
    text=True,
)
print({'gpu_probe_return_code': gpu_probe.returncode, 'gpu_names': gpu_probe.stdout.strip()})
assert gpu_probe.returncode == 0 and gpu_probe.stdout.strip(), '需要 Colab GPU runtime'


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

workflow_route = 'external_baseline_shallow_diffuse'
result_path = Path("outputs/formal_workflow_execution") / SLM_WM_PAPER_RUN_NAME / SLM_WM_RANDOMIZATION_REPEAT_ID / workflow_route / "workflow_result.json"
host_command = [
    sys.executable,
    "-I",
    FORMAL_HOST_LAUNCHER,
    "--root",
    ".",
    "--repository-commit",
    repository_commit,
    "gpu",
    "--workflow",
    workflow_route,
    "--paper-run-name",
    SLM_WM_PAPER_RUN_NAME,
    "--randomization-repeat-id",
    SLM_WM_RANDOMIZATION_REPEAT_ID,
    "--result-path",
    result_path.as_posix(),
]
subprocess.run(host_command, check=True)
formal_workflow_result = json.loads(result_path.read_text(encoding="utf-8"))
assert formal_workflow_result["decision"] == "pass", formal_workflow_result
formal_workflow_result


In [ ]:
print(json.dumps(formal_workflow_result, ensure_ascii=False, sort_keys=True, indent=2))
